In [1]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

In [3]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/KaburAjaDulu/data/cleaned_data.csv")
docs = df["clean_text"].astype(str).tolist()

Mounted at /content/drive


In [6]:
with open(
    "/content/drive/MyDrive/KaburAjaDulu/data/combined_stop_words.txt",
    "r",
    encoding="utf-8"
) as f:

    stopwords_id = [
        line.strip()
        for line in f
        if line.strip()
    ]

In [7]:
extra_stopwords = [
    'kaburajadulu',
    'kabur',
    'dulu',
    'aja',
    'yg',
    'ya',
    'ga',
    'gak',
    'nggak',
    'si',
    'lo',
    'gw',
    'lu',
    'bang',
    'kak',
    'deh',
    'sih',
    'nih',
    'lah',
    'dong',
    'kan',
    'kalo',
    'udah',
    'udh',
    'kayak',
    'kayaknya',
    'mah',
    'wkwk',
    'wkwkwk',
    'haha',
    'hehe',
    'btw',
    'fyi',
    'dll',
    'dsb',
    'yuk',
    'tuh'
]

In [8]:
all_stopwords = list(
    set(
        stopwords_id +
        extra_stopwords
    )
)

In [9]:
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [11]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [12]:
vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    min_df=5,
    ngram_range=(1,2)
)

In [13]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True
)

In [14]:
topics, probs = topic_model.fit_transform(
    docs
)

2026-06-15 15:08:52,998 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/588 [00:00<?, ?it/s]

2026-06-15 15:09:06,933 - BERTopic - Embedding - Completed ✓
2026-06-15 15:09:06,934 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-15 15:09:51,082 - BERTopic - Dimensionality - Completed ✓
2026-06-15 15:09:51,084 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-15 15:09:57,872 - BERTopic - Cluster - Completed ✓
2026-06-15 15:09:57,884 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-15 15:09:58,545 - BERTopic - Representation - Completed ✓


In [15]:
df["topic_id_v2"] = topics

df.head()

,created_at,full_text,id_str,platform,clean_text,topic_id_v2
0,Tue Sep 30 23:48:17 +0000 2025,EMANG BNER GUE #KaburAjaDulu ATP,1973173082253828399,X,emang bner saya kaburajadulu atp,55
1,Tue Sep 30 19:41:13 +0000 2025,Wkwkwkw masih aja. Saran saya saatnya #KaburAj...,1973110905211855020,X,wkwkwkw masih saja saran saya saatnya kaburaja...,0
2,Tue Sep 30 15:25:47 +0000 2025,Selamat dan sukses kepada siswa-siswi LPK Yosh...,1973046626290897319,X,selamat dan sukses kepada siswa siswi lpk yosh...,4
3,Tue Sep 30 14:43:04 +0000 2025,@alestarbluu Sumpahh aku pro ke statement #kab...,1973035876054905276,X,sumpahh aku profesional ke statement kaburajad...,0
4,Tue Sep 30 12:47:18 +0000 2025,Manual order shipped safely️ bukunya #KaburAja...,1973006742117298281,X,manual order shipped safely bukunya kaburajadu...,9


In [16]:
topic_info = topic_model.get_topic_info()

topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5316,-1_kerja_iya_negeri_nya,"[kerja, iya, negeri, nya, negri, banget, anak,...","[belajar yang pinter biar bisa pindah negara, ..."
1,0,2605,0_nasionalis_dpr_hastag_rakyat,"[nasionalis, dpr, hastag, rakyat, anggota, iya...",[katanya wakil rakyat rakyat bikin gerakan kab...
2,1,1908,1_percaya_pemerintah_percaya pemerintah_rakyat,"[percaya, pemerintah, percaya pemerintah, raky...","[sudah ga percaya sama pemerintah, saya ga per..."
3,2,1084,2_indonesia_indonesiagelap_gelap_cinta,"[indonesia, indonesiagelap, gelap, cinta, indo...","[ini indonesia booyyy bukan sgp, indonesia cem..."
4,3,869,3_kerja negeri_negeri_kerja_negeri kerja,"[kerja negeri, negeri, kerja, negeri kerja, pe...","[kerja luar negeri sudah ga ldr lagi, kerja di..."
5,4,451,4_jepang_bahasa_belajar_lpk,"[jepang, bahasa, belajar, lpk, belajar bahasa,...","[dan saya sudah kabur ke jepang, yoo kita ke j..."
6,5,349,5_indo_kerja indo_indo kerja_kerja,"[indo, kerja indo, indo kerja, kerja, gaji, or...","[indo mulai berantakan, paling sebel ormasnya ..."
7,6,326,6_nasionalisme_meragukan_nasionalis_bahlil,"[nasionalisme, meragukan, nasionalis, bahlil, ...",[nasionalisme gak bisa menunjang hidup kami pa...
8,7,234,7_gaji_jt_gajinya_bercanda,"[gaji, jt, gajinya, bercanda, gaji jt, serius,...",[saya di jakartaaa leader gudang gaji jutaaaaa...
9,8,203,8_malaysia_singapore_malay_singapura,"[malaysia, singapore, malay, singapura, susant...","[ke malaysia, jangan jauh jauh padahal kaburaj..."


In [17]:
total_topics = len(
    topic_info[topic_info["Topic"] != -1]
)
noise_count = len(
    df[df["topic_id_v2"] == -1]
)
valid_count = len(
    df[df["topic_id_v2"] != -1]
)

In [18]:
print(f"Jumlah topik: {total_topics}")

print(f"Noise: {noise_count}")

print(f"Valid: {valid_count}")

Jumlah topik: 90
Noise: 5316
Valid: 13498


In [19]:
topic_info[["Topic","Count","Name"]].head(20)

,Topic,Count,Name
0,-1,5316,-1_kerja_iya_negeri_nya
1,0,2605,0_nasionalis_dpr_hastag_rakyat
2,1,1908,1_percaya_pemerintah_percaya pemerintah_rakyat
3,2,1084,2_indonesia_indonesiagelap_gelap_cinta
4,3,869,3_kerja negeri_negeri_kerja_negeri kerja
5,4,451,4_jepang_bahasa_belajar_lpk
6,5,349,5_indo_kerja indo_indo kerja_kerja
7,6,326,6_nasionalisme_meragukan_nasionalis_bahlil
8,7,234,7_gaji_jt_gajinya_bercanda
9,8,203,8_malaysia_singapore_malay_singapura


In [20]:
topic_info.to_csv(
    "bertopic_info_v2.csv",
    index=False)

df.to_csv(
    "data_with_topics_v2.csv",
    index=False)